In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "آبی"
    least_favourite_colour: str = "زرد"

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColourContext  
)

In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="رنگ مورد علاقه من چیه؟")]},
    context=ColourContext()
)

In [8]:
print(response["messages"][-1].content)

نمیتونم دقیقاً بدون اطلاعی از تو بگم، اما می‌تونم با چند سوال کوچک حدس بزنم. آماده‌ای؟

سوال‌ها (جواب بده تا حدسم رو بگم):
1) ترجیح می‌دی رنگ‌هایWarm (قرمز/نارنجی/زرد) باشن یا Cool (آبی/سبز/بنفش)؟  
2) دوست داری رنگ روشن باشه یا تیره؟  
3) بیشتر رنگ رو در لباس دوست داری یا در دکور و فضای اطرافت؟  
4) رنگی هست که به طور خاص ازش خوش‌ت نمی‌آید؟

بعد از پاسخ‌ها، یکی از رنگ‌های رایج رو بهت حدس می‌زنم. یا اگر می‌خواهی همین الان یک حدس سریع بخواهی، بگو تا با یک سوال حدس بزنم.


## Accessing Context

In [13]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [15]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [17]:
response = agent.invoke(
    {"messages": [HumanMessage(content="رنگ مورد علاقه ام چیه؟")]},
    context=ColourContext()
)

print(response["messages"][-1].content)

C:\Users\Alireza\AppData\Roaming\Python\Python312\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='زرد'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
C:\Users\Alireza\AppData\Roaming\Python\Python312\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='زرد'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


رنگ مورد علاقه‌تان آبی است. دوست دارید این را ثبت کنم یا تغییر بدهیم؟ اگر باز هم رنگ دیگری را هم اضافه کنید می‌توانم بگویم.


In [19]:
response = agent.invoke(
    {"messages": [HumanMessage(content="رنگ مورد علاقه ام چیه؟")]},
    context=ColourContext(favourite_colour="green")
)

print(response["messages"][-1].content)

C:\Users\Alireza\AppData\Roaming\Python\Python312\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='زرد'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
C:\Users\Alireza\AppData\Roaming\Python\Python312\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='زرد'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


رنگ مورد علاقه‌ت سبز است.

اگر بخواهی، می‌تونم دربارهٔ سایه‌های مختلف سبز یا ترکیب‌های رنگی با سبز هم پیشنهاد بدم.


## Dynamic System Prompt

In [28]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from typing import TypedDict

class UserContext(TypedDict):
    user_level: str  # "beginner" یا "expert"


In [30]:
@dynamic_prompt
def adaptive_prompt(request: ModelRequest) -> str:
    """System prompt بر اساس سطح کاربر"""
    level = request.runtime.context.get("user_level", "beginner")
    
    if level == "expert":
        return "You are a technical assistant. Use precise terminology and assume advanced knowledge."
    else:
        return "You are a friendly assistant. Explain concepts simply and avoid jargon."


In [32]:
agent_adaptive = create_agent(
    model="gpt-5-nano",
    middleware=[adaptive_prompt],
    context_schema=UserContext,
)

In [34]:
# برای مبتدی
response_beginner = agent_adaptive.invoke(
    {"messages": [{"role": "user", "content": "What is machine learning?"}]},
    context={"user_level": "beginner"}
)
print("For beginner:")
print(response_beginner["messages"][-1].content)


For beginner:
Machine learning is a way to get computers to do tasks by learning from data, rather than being told every rule exactly.

- How it works (in simple terms): You give the computer lots of examples with the correct answers, and it learns patterns in the data. Then it uses what it learned to make guesses on new, unseen data.
- Types (very roughly):
  - Supervised learning: learn from labeled examples (e.g., emails labeled “spam” or “not spam”).
  - Unsupervised learning: find patterns in unlabeled data (e.g., grouping customers by similar behavior).
  - Reinforcement learning: learn by trying actions and getting feedback (rewards or penalties) from the environment.
- Common examples: spam filters, movie or product recommendations, voice or handwriting recognition, predicting house prices.
- The idea in practice: collect data, train a model on that data, test how well it works, and use the model to make predictions on new data.
- Why it’s useful and its limits: it can handle l

In [36]:
# برای متخصص
response_expert = agent_adaptive.invoke(
    {"messages": [{"role": "user", "content": "What is machine learning?"}]},
    context={"user_level": "expert"}
)
print("For expert:")
print(response_expert["messages"][-1])


For expert:
content='Machine learning is the field of study that gives computers the ability to learn from data and make predictions or decisions without being explicitly programmed for every task. In practice, it involves fitting a model that maps inputs to outputs by optimizing a loss function over data.\n\nKey ideas:\n- Learning from data: adjust model parameters to improve performance on observed examples.\n- Model and objective: a hypothesis space (e.g., linear models, trees, neural networks) and a loss function (e.g., cross-entropy, mean squared error) guide training.\n- Optimization: use algorithms like gradient descent to minimize the loss on training data.\n- Generalization: the goal is good performance on unseen data, not just on the training set.\n\nCommon problem types:\n- Supervised learning: learn from labeled pairs (x, y) for tasks like classification and regression.\n- Unsupervised learning: discover structure in data (e.g., clustering, dimensionality reduction).\n- Sem